# Huấn luyện baseline

## Đọc dữ liệu

Đọc NHANES 2015–2018, lấy đúng 8 đầu vào làm X và `diabetes_label` làm y. Giữ `SEQN`, `cycle` và các cột khảo sát riêng, khớp index với X/y; số đếm và metric trong notebook chưa áp trọng số khảo sát.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold

nhanes = pd.read_parquet(Path("../data/processed/nhanes_2015_2018.parquet"))
input_cols = [
    "age", "sex", "race_ethnicity", "bmi", "smoking_status",
    "physical_activity", "heart_attack", "stroke",
]
survey_cols = ["WTINT2YR", "WTMEC2YR", "WTSAF2YR", "SDMVPSU", "SDMVSTRA"]
X = nhanes[input_cols].copy()
y = nhanes["diabetes_label"].copy()
support = nhanes[["SEQN", "cycle"] + survey_cols].copy()
strata = (y.astype("string") + "|" + support["cycle"].astype("string")).rename("strata")

assert X.shape == (10832, 8)
assert X.columns.tolist() == input_cols
assert X.index.equals(y.index) and X.index.equals(support.index)
assert support["SEQN"].notna().all() and support["SEQN"].is_unique
assert support["cycle"].notna().all()
assert y.notna().all() and y.isin([0, 1]).all()
assert strata.nunique() == 4
pd.DataFrame({
    "Số người": [len(X)], "Số đầu vào": [X.shape[1]],
    "Nhãn 0": [int(y.eq(0).sum())], "Nhãn 1": [int(y.eq(1).sum())],
    "Số strata": [strata.nunique()],
})

,Số người,Số đầu vào,Nhãn 0,Nhãn 1,Số strata
0,10832,8,8558,2274,4


## Chia fold

Thiết kế nested CV gồm 5 fold ngoài và 5 fold trong, không thêm holdout. Vòng trong dành cho chọn mô hình/siêu tham số; vòng ngoài đánh giá quy trình trên người không dùng để fit. Các baseline bên dưới chỉ dùng 5 fold ngoài với cấu hình cố định, chưa chạy tối ưu ở vòng trong.

Cả hai vòng dùng `StratifiedKFold(shuffle=True, random_state=42)`, chia theo tổ hợp nhãn bệnh × chu kỳ; nhãn huấn luyện vẫn là y. Bốn mô hình dùng chung các bộ chỉ số đã lưu.

`outer_splits` là vị trí dòng trong X/y/support, dùng với `.iloc`. `inner_splits_by_outer` là vị trí tương đối trong outer-train tương ứng, không phải vị trí trong toàn bộ X.

In [2]:
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
outer_splits = [(train_idx.copy(), test_idx.copy())
                for train_idx, test_idx in outer_cv.split(X, strata)]
inner_splits_by_outer = {}
for fold, (outer_train_idx, outer_test_idx) in enumerate(outer_splits, start=1):
    inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    X_outer_train = X.iloc[outer_train_idx]
    strata_outer_train = strata.iloc[outer_train_idx]
    inner_splits_by_outer[fold] = [
        (train_idx.copy(), valid_idx.copy())
        for train_idx, valid_idx in inner_cv.split(X_outer_train, strata_outer_train)
    ]

In [3]:
outer_test_coverage = np.zeros(len(X), dtype=int)
fold_checks = []
for fold, (outer_train_idx, outer_test_idx) in enumerate(outer_splits, start=1):
    assert len(np.intersect1d(outer_train_idx, outer_test_idx)) == 0
    assert np.array_equal(np.sort(np.concatenate([outer_train_idx, outer_test_idx])), np.arange(len(X)))
    train_seqn = set(support.iloc[outer_train_idx]["SEQN"])
    test_seqn = set(support.iloc[outer_test_idx]["SEQN"])
    assert train_seqn.isdisjoint(test_seqn)
    outer_test_coverage[outer_test_idx] += 1

    inner_valid_coverage = np.zeros(len(outer_train_idx), dtype=int)
    assert len(inner_splits_by_outer[fold]) == 5
    for inner_train_idx, inner_valid_idx in inner_splits_by_outer[fold]:
        assert np.array_equal(
            np.sort(np.concatenate([inner_train_idx, inner_valid_idx])),
            np.arange(len(outer_train_idx)),
        )
        assert len(np.intersect1d(inner_train_idx, inner_valid_idx)) == 0
        # Đổi chỉ số tương đối sang vị trí trong toàn bộ X chỉ để kiểm tra.
        inner_train_global = outer_train_idx[inner_train_idx]
        inner_valid_global = outer_train_idx[inner_valid_idx]
        assert set(inner_train_global).issubset(set(outer_train_idx))
        assert set(inner_valid_global).issubset(set(outer_train_idx))
        assert len(np.intersect1d(inner_train_global, outer_test_idx)) == 0
        assert len(np.intersect1d(inner_valid_global, outer_test_idx)) == 0
        assert set(support.iloc[inner_train_global]["SEQN"]).isdisjoint(
            set(support.iloc[inner_valid_global]["SEQN"])
        )
        inner_valid_coverage[inner_valid_idx] += 1
    assert np.all(inner_valid_coverage == 1)
    fold_checks.append({
        "Fold ngoài": fold, "Outer-train": len(outer_train_idx),
        "Outer-test": len(outer_test_idx), "Số fold trong": 5,
        "Không giao người": True,
    })
assert len(outer_splits) == 5
assert np.all(outer_test_coverage == 1)
pd.DataFrame(fold_checks)

,Fold ngoài,Outer-train,Outer-test,Số fold trong,Không giao người
0,1,8665,2167,5,True
1,2,8665,2167,5,True
2,3,8666,2166,5,True
3,4,8666,2166,5,True
4,5,8666,2166,5,True


## Số người theo fold

Đếm người và nhãn 0/1 theo chu kỳ trong train/test của từng fold ngoài.

In [4]:
fold_count_tables = []
for fold, (train_idx, test_idx) in enumerate(outer_splits, start=1):
    for split_name, positions in [("Train", train_idx), ("Test", test_idx)]:
        table = pd.crosstab(support.iloc[positions]["cycle"], y.iloc[positions])
        table = table.reindex(columns=[0, 1], fill_value=0).rename(columns={0: "Nhãn 0", 1: "Nhãn 1"})
        table.columns.name = None
        table.index.name = "Chu kỳ"
        table.insert(0, "Số người", table.sum(axis=1))
        table = table.reset_index()
        table.insert(0, "Tập", split_name)
        table.insert(0, "Fold ngoài", fold)
        fold_count_tables.append(table)
outer_fold_counts = pd.concat(fold_count_tables, ignore_index=True)
assert outer_fold_counts["Số người"].eq(outer_fold_counts["Nhãn 0"] + outer_fold_counts["Nhãn 1"]).all()
assert outer_fold_counts.groupby("Fold ngoài")["Số người"].sum().eq(10832).all()
outer_fold_counts

,Fold ngoài,Tập,Chu kỳ,Số người,Nhãn 0,Nhãn 1
0,1,Train,2015-2016,4401,3517,884
1,1,Train,2017-2018,4264,3329,935
2,1,Test,2015-2016,1101,880,221
3,1,Test,2017-2018,1066,832,234
4,2,Train,2015-2016,4401,3517,884
5,2,Train,2017-2018,4264,3329,935
6,2,Test,2015-2016,1101,880,221
7,2,Test,2017-2018,1066,832,234
8,3,Train,2015-2016,4402,3518,884
9,3,Train,2017-2018,4264,3328,936


Mỗi outer-test có 2.166–2.167 người; mỗi người thuộc test đúng một lần và không có `SEQN` giao giữa train/test. Cả 25 bộ chia vòng trong chỉ nằm trong outer-train tương ứng.

Mọi bước học từ dữ liệu phải fit riêng trên train của fold đang xét, rồi mới áp dụng cho validation/test.

## Kiểm tra tiền xử lý

`age`, `bmi` được điền trung vị; sáu biến phân loại được điền giá trị thường gặp nhất rồi one-hot với `handle_unknown="ignore", drop=None`. Chỉ LR/SVM dùng thêm `StandardScaler` cho hai biến liên tục; không thêm cờ thiếu hay winsorize.

Fit thử trên outer-train của fold 1 rồi transform train/test bằng cùng đối tượng, để median, mode và scaling chỉ học từ train. `pd.NA` được đổi sang `np.nan` trong bản sao nội bộ, không sửa X. Các bản `_demo` chỉ để kiểm tra; mỗi fold khi huấn luyện phải tạo pipeline mới.

In [5]:
import sys
from importlib import reload

if ".." not in sys.path:
    sys.path.insert(0, "..")
import src.utils as utils
reload(utils)
from src.utils import build_preprocessor

X_before_preprocessor_demo = X.copy(deep=True)
demo_train_idx, demo_test_idx = outer_splits[0]
X_train_demo = X.iloc[demo_train_idx].copy()
X_test_demo = X.iloc[demo_test_idx].copy()
preprocessor_scaled_demo = build_preprocessor(scale_numeric=True)
preprocessor_unscaled_demo = build_preprocessor(scale_numeric=False)
transformed_demo = {}
transform_shapes = []
for name, preprocessor in [
    ("LR/SVM: có scaling", preprocessor_scaled_demo),
    ("RF/XGBoost: không scaling", preprocessor_unscaled_demo),
]:
    train_values = preprocessor.fit_transform(X_train_demo)
    names_before_test = preprocessor.get_feature_names_out().copy()
    test_values = preprocessor.transform(X_test_demo)
    np.testing.assert_array_equal(names_before_test, preprocessor.get_feature_names_out())
    assert train_values.shape[1] == test_values.shape[1] == len(names_before_test)
    assert np.isfinite(train_values).all() and np.isfinite(test_values).all()
    transformed_demo[name] = (train_values, test_values)
    transform_shapes.append({
        "Bản kiểm tra": name,
        "Train sau biến đổi": train_values.shape,
        "Test sau biến đổi": test_values.shape,
        "Hữu hạn, không thiếu": True,
    })
pd.DataFrame(transform_shapes)

,Bản kiểm tra,Train sau biến đổi,Test sau biến đổi,"Hữu hạn, không thiếu"
0,LR/SVM: có scaling,"(8665, 17)","(2167, 17)",True
1,RF/XGBoost: không scaling,"(8665, 17)","(2167, 17)",True


In [6]:
numeric_cols = ["age", "bmi"]
train_medians = X_train_demo[numeric_cols].median()
scaled_numeric_demo = preprocessor_scaled_demo.named_transformers_["numeric"]
unscaled_numeric_demo = preprocessor_unscaled_demo.named_transformers_["numeric"]
for numeric_pipeline in [scaled_numeric_demo, unscaled_numeric_demo]:
    np.testing.assert_allclose(numeric_pipeline.named_steps["imputer"].statistics_, train_medians)

# Mean/scale được học sau khi điền median của riêng train.
filled_train_numeric = X_train_demo[numeric_cols].astype(float).fillna(train_medians)
scaler_demo = scaled_numeric_demo.named_steps["scaler"]
expected_scale = filled_train_numeric.std(ddof=0).replace(0, 1)
np.testing.assert_allclose(scaler_demo.mean_, filled_train_numeric.mean())
np.testing.assert_allclose(scaler_demo.scale_, expected_scale)
pd.DataFrame({
    "Median riêng train": train_medians,
    "Median đã học": scaled_numeric_demo.named_steps["imputer"].statistics_,
    "Scaler mean": scaler_demo.mean_,
    "Scaler scale": scaler_demo.scale_,
})

,Median riêng train,Median đã học,Scaler mean,Scaler scale
age,50.0,50.0,49.207617,18.548138
bmi,28.4,28.4,29.552856,7.153647


In [7]:
categorical_cols = [
    "sex", "race_ethnicity", "smoking_status",
    "physical_activity", "heart_attack", "stroke",
]
train_modes = [X_train_demo[column].mode(dropna=True).iloc[0] for column in categorical_cols]
scaled_modes = preprocessor_scaled_demo.named_transformers_["categorical"].named_steps["imputer"].statistics_
unscaled_modes = preprocessor_unscaled_demo.named_transformers_["categorical"].named_steps["imputer"].statistics_
assert list(scaled_modes) == list(unscaled_modes) == train_modes
pd.DataFrame({
    "Biến": categorical_cols, "Mode riêng train": train_modes, "Mode đã học": scaled_modes,
})

,Biến,Mode riêng train,Mode đã học
0,sex,2,2
1,race_ethnicity,White NH,White NH
2,smoking_status,never,never
3,physical_activity,0,0
4,heart_attack,0,0
5,stroke,0,0


In [8]:
feature_names_demo = preprocessor_scaled_demo.get_feature_names_out()
np.testing.assert_array_equal(
    feature_names_demo, preprocessor_unscaled_demo.get_feature_names_out()
)
pd.DataFrame({
    "Thứ tự": np.arange(1, len(feature_names_demo) + 1),
    "Đặc trưng": feature_names_demo,
})

,Thứ tự,Đặc trưng
0,1,numeric__age
1,2,numeric__bmi
2,3,categorical__sex_1
3,4,categorical__sex_2
4,5,categorical__race_ethnicity_Black NH
5,6,categorical__race_ethnicity_Hispanic
6,7,categorical__race_ethnicity_Other
7,8,categorical__race_ethnicity_White NH
8,9,categorical__smoking_status_current
9,10,categorical__smoking_status_former


In [9]:
pd.testing.assert_frame_equal(X, X_before_preprocessor_demo)
pd.testing.assert_frame_equal(X_train_demo, X.iloc[demo_train_idx])
pd.testing.assert_frame_equal(X_test_demo, X.iloc[demo_test_idx])
assert "scaler" not in unscaled_numeric_demo.named_steps
for preprocessor in [preprocessor_scaled_demo, preprocessor_unscaled_demo]:
    encoder = preprocessor.named_transformers_["categorical"].named_steps["onehot"]
    assert encoder.drop is None and encoder.handle_unknown == "ignore"
# Mỗi lần gọi hàm phải tạo cả transformer và các bước con mới, chưa fit.
fresh_preprocessor_demo = build_preprocessor(scale_numeric=True)
assert fresh_preprocessor_demo is not preprocessor_scaled_demo
assert not hasattr(fresh_preprocessor_demo, "transformers_")
assert fresh_preprocessor_demo.transformers[0][1] is not preprocessor_scaled_demo.transformers[0][1]
pd.Series({
    "Median/mode chỉ từ train": True,
    "Mean/scale chỉ từ train đã điền median": True,
    "Train/test cùng số cột và thứ tự": True,
    "X gốc không đổi": True,
    "Hàm tạo đối tượng mới, chưa fit": True,
}, name="Đạt kiểm tra").to_frame()

,Đạt kiểm tra
Median/mode chỉ từ train,True
Mean/scale chỉ từ train đã điền median,True
Train/test cùng số cột và thứ tự,True
X gốc không đổi,True
"Hàm tạo đối tượng mới, chưa fit",True


Hai bản đều có 17 đặc trưng: 2 biến liên tục và 15 cột one-hot. Train/test lần lượt có 8.665/2.167 người, đầu ra hữu hạn và không thiếu; median/mode khớp với riêng outer-train.

## LR baseline

Dùng `C=1`, `solver="lbfgs"`, `max_iter=2000`, `random_state=42` và `class_weight="balanced"` tính từ y_train của từng fold. Mỗi fold tạo pipeline mới có scaling.

Thu xác suất nhãn 1 ngoài fold, dùng cho ROC-AUC/AP, Brier và phân loại với ngưỡng cố định 0,5. Xác suất từ mô hình có trọng số lớp chưa được hiệu chỉnh bổ sung.

In [10]:
import warnings
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.exceptions import ConvergenceWarning
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    roc_auc_score, average_precision_score, recall_score,
    precision_score, f1_score, brier_score_loss, confusion_matrix,
)

X_before_lr_baseline = X.copy(deep=True)
y_before_lr_baseline = y.copy(deep=True)
lr_oof_probability = pd.Series(np.nan, index=X.index, name="probability_1")
lr_oof_fold = pd.Series(pd.NA, index=X.index, dtype="Int64", name="outer_fold")
lr_oof_coverage = np.zeros(len(X), dtype=int)
lr_baseline_models = {}
lr_fold_rows = []
lr_warning_rows = []
lr_weight_rows = []
for fold, (train_idx, test_idx) in enumerate(outer_splits, start=1):
    X_train = X.iloc[train_idx]
    y_train = y.iloc[train_idx]
    X_test = X.iloc[test_idx]
    y_test = y.iloc[test_idx]
    model = Pipeline([
        ("preprocessor", build_preprocessor(scale_numeric=True)),
        ("classifier", LogisticRegression(
            C=1.0, class_weight="balanced", solver="lbfgs",
            max_iter=2000, random_state=42,
        )),
    ])
    assert model.named_steps["preprocessor"] is not preprocessor_scaled_demo
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        model.fit(X_train, y_train)
    for warning in caught:
        lr_warning_rows.append({
            "Fold": fold, "Loại": warning.category.__name__,
            "Thông báo": str(warning.message),
            "Không hội tụ": issubclass(warning.category, ConvergenceWarning),
        })
    weights = compute_class_weight("balanced", classes=np.array([0, 1]), y=y_train.to_numpy(dtype=int))
    lr_weight_rows.append({
        "Fold": fold, "Train nhãn 0": int(y_train.eq(0).sum()),
        "Train nhãn 1": int(y_train.eq(1).sum()),
        "Trọng số lớp 0": weights[0], "Trọng số lớp 1": weights[1],
    })
    np.testing.assert_allclose(weights, [len(y_train) / (2 * y_train.eq(label).sum()) for label in [0, 1]])
    positive_column = int(np.flatnonzero(model.classes_ == 1)[0])
    probability = model.predict_proba(X_test)[:, positive_column]
    prediction = (probability >= 0.5).astype(int)
    assert np.isfinite(probability).all() and ((probability >= 0) & (probability <= 1)).all()
    lr_oof_probability.iloc[test_idx] = probability
    lr_oof_fold.iloc[test_idx] = fold
    lr_oof_coverage[test_idx] += 1
    tn, fp, fn, tp = confusion_matrix(y_test, prediction, labels=[0, 1]).ravel()
    lr_fold_rows.append({
        "Fold": fold,
        "ROC-AUC": roc_auc_score(y_test, probability),
        "Average precision": average_precision_score(y_test, probability),
        "Sensitivity": recall_score(y_test, prediction, zero_division=0),
        "Specificity": tn / (tn + fp),
        "Precision": precision_score(y_test, prediction, zero_division=0),
        "F1": f1_score(y_test, prediction, zero_division=0),
        "Brier score": brier_score_loss(y_test, probability),
    })
    lr_baseline_models[fold] = model
lr_fold_metrics = pd.DataFrame(lr_fold_rows).set_index("Fold")
lr_fold_metrics.round(4)

,ROC-AUC,Average precision,Sensitivity,Specificity,Precision,F1,Brier score
Fold,,,,,,,
1,0.7789,0.4383,0.7451,0.6665,0.3725,0.4967,0.1977
2,0.7932,0.4698,0.7736,0.6846,0.3946,0.5226,0.1899
3,0.7981,0.4575,0.7819,0.6887,0.3998,0.5291,0.1901
4,0.7935,0.4599,0.7495,0.6856,0.3879,0.5112,0.1869
5,0.8128,0.4953,0.7978,0.6973,0.4120,0.5434,0.1810


In [11]:
lr_metric_summary = lr_fold_metrics.agg(["mean", "std"]).rename(
    index={"mean": "Trung bình", "std": "Độ lệch chuẩn"}
)
lr_metric_summary.round(4)

,ROC-AUC,Average precision,Sensitivity,Specificity,Precision,F1,Brier score
Trung bình,0.7953,0.4642,0.7696,0.6845,0.3934,0.5206,0.1891
Độ lệch chuẩn,0.0121,0.0208,0.0222,0.0112,0.0146,0.0177,0.0060


In [12]:
assert lr_oof_probability.index.equals(X.index) and lr_oof_probability.index.equals(y.index)
assert lr_oof_probability.notna().all() and lr_oof_fold.notna().all()
assert np.isfinite(lr_oof_probability).all()
assert lr_oof_probability.between(0, 1).all()
assert np.all(lr_oof_coverage == 1)
lr_oof_prediction = lr_oof_probability.ge(0.5).astype("Int64").rename("prediction")
lr_oof = pd.DataFrame({
    "SEQN": support["SEQN"], "cycle": support["cycle"],
    "y_true": y, "probability_1": lr_oof_probability,
    "prediction": lr_oof_prediction, "outer_fold": lr_oof_fold,
})
assert lr_oof.index.equals(X.index) and lr_oof["SEQN"].is_unique
assert len(lr_oof) == 10832
for fold, (_, test_idx) in enumerate(outer_splits, start=1):
    assert lr_oof_fold.iloc[test_idx].eq(fold).all()
    assert set(lr_oof.loc[lr_oof["outer_fold"].eq(fold), "SEQN"]) == set(support.iloc[test_idx]["SEQN"])
pd.DataFrame(
    confusion_matrix(y, lr_oof_prediction, labels=[0, 1]),
    index=pd.Index(["0", "1"], name="Nhãn thật"),
    columns=pd.Index(["0", "1"], name="Dự đoán (ngưỡng 0,5)"),
)

"Dự đoán (ngưỡng 0,5)",0,1
Nhãn thật,,
0,5858,2700
1,524,1750


In [13]:
pd.testing.assert_frame_equal(X, X_before_lr_baseline)
pd.testing.assert_series_equal(y, y_before_lr_baseline)
assert len({id(m.named_steps["preprocessor"]) for m in lr_baseline_models.values()}) == 5
assert len({id(m.named_steps["classifier"]) for m in lr_baseline_models.values()}) == 5
lr_baseline_warnings = pd.DataFrame(lr_warning_rows, columns=["Fold", "Loại", "Thông báo", "Không hội tụ"])
lr_baseline_class_weights = pd.DataFrame(lr_weight_rows).set_index("Fold")
lr_convergence_warnings = sum(row["Không hội tụ"] for row in lr_warning_rows)
pd.Series({
    "Số dự đoán ngoài fold": len(lr_oof_probability),
    "Dự đoán thiếu": int(lr_oof_probability.isna().sum()),
    "Mỗi người được dự đoán đúng một lần": bool(np.all(lr_oof_coverage == 1)),
    "X/y giữ nguyên": True,
    "Số cảnh báo": len(lr_baseline_warnings),
    "Cảnh báo không hội tụ": lr_convergence_warnings,
}, name="Kiểm tra").to_frame()

,Kiểm tra
Số dự đoán ngoài fold,10832
Dự đoán thiếu,0
Mỗi người được dự đoán đúng một lần,True
X/y giữ nguyên,True
Số cảnh báo,0
Cảnh báo không hội tụ,0


In [14]:
lr_baseline_warnings

,Fold,Loại,Thông báo,Không hội tụ


Confusion matrix gộp dự đoán ngoài fold, mỗi người đúng một lần. Trung bình ± SD tính giữa 5 fold với `ddof=1`; SD không phải khoảng tin cậy.

## XGBoost baseline

Dùng cùng X, y và fold của LR, với 200 cây, độ sâu 3 và learning rate 0,05; các tham số cố định còn lại ghi trong code. Đây là cấu hình baseline của dự án, không khẳng định là cấu hình tối ưu hay cấu hình của paper.

Mỗi fold tạo pipeline mới không scaling; `scale_pos_weight = số nhãn 0 / số nhãn 1` chỉ từ y_train. Không thêm `sample_weight`, early stopping hay `eval_set`; phân loại bằng xác suất ≥0,5, chưa hiệu chỉnh bổ sung.

LR và XGBoost dùng cùng kernel `NHANES-BRFSS (.venv)`; phiên bản thư viện được ghi ở ô dưới.

In [15]:
import sys
from time import perf_counter
import sklearn
import xgboost
from xgboost import XGBClassifier

xgb_environment = pd.Series({
    "Python": sys.executable,
    "xgboost": xgboost.__version__,
    "scikit-learn": sklearn.__version__,
}, name="Môi trường")
xgb_environment.to_frame()

,Môi trường
Python,D:\NCKH Mini\.venv\Scripts\python.exe
xgboost,3.4.1
scikit-learn,1.9.1


In [16]:
X_before_xgb_baseline = X.copy(deep=True)
y_before_xgb_baseline = y.copy(deep=True)
xgb_oof_probability = pd.Series(np.nan, index=X.index, name="probability_1")
xgb_oof_fold = pd.Series(pd.NA, index=X.index, dtype="Int64", name="outer_fold")
xgb_oof_coverage = np.zeros(len(X), dtype=int)
xgb_baseline_models = {}
xgb_fold_rows = []
xgb_fit_rows = []
xgb_warning_rows = []
for fold, (train_idx, test_idx) in enumerate(outer_splits, start=1):
    X_train = X.iloc[train_idx]
    y_train = y.iloc[train_idx]
    X_test = X.iloc[test_idx]
    y_test = y.iloc[test_idx]
    n_negative = int(y_train.eq(0).sum())
    n_positive = int(y_train.eq(1).sum())
    assert n_positive > 0
    scale_pos_weight = n_negative / n_positive
    model = Pipeline([
        ("preprocessor", build_preprocessor(scale_numeric=False)),
        ("classifier", XGBClassifier(
            objective="binary:logistic", eval_metric="logloss",
            n_estimators=200, max_depth=3, learning_rate=0.05,
            min_child_weight=1, subsample=0.8, colsample_bytree=0.8,
            reg_alpha=0, reg_lambda=1, tree_method="hist",
            random_state=42, n_jobs=4, scale_pos_weight=scale_pos_weight,
        )),
    ])
    assert model.named_steps["preprocessor"] is not preprocessor_unscaled_demo
    assert model.named_steps["preprocessor"] is not lr_baseline_models[fold].named_steps["preprocessor"]
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        fit_start = perf_counter()
        model.fit(X_train, y_train)
        fit_seconds = perf_counter() - fit_start
    for warning in caught:
        xgb_warning_rows.append({
            "Fold": fold, "Loại": warning.category.__name__, "Thông báo": str(warning.message),
        })
    classifier = model.named_steps["classifier"]
    assert classifier.get_params()["scale_pos_weight"] == n_negative / n_positive
    assert classifier.get_params().get("early_stopping_rounds") is None
    assert classifier.get_booster().num_boosted_rounds() == 200
    assert "scaler" not in model.named_steps["preprocessor"].named_transformers_["numeric"].named_steps
    positive_column = int(np.flatnonzero(model.classes_ == 1)[0])
    probability = model.predict_proba(X_test)[:, positive_column]
    prediction = (probability >= 0.5).astype(int)
    assert np.isfinite(probability).all() and ((probability >= 0) & (probability <= 1)).all()
    xgb_oof_probability.iloc[test_idx] = probability
    xgb_oof_fold.iloc[test_idx] = fold
    xgb_oof_coverage[test_idx] += 1
    tn, fp, fn, tp = confusion_matrix(y_test, prediction, labels=[0, 1]).ravel()
    xgb_fold_rows.append({
        "Fold": fold,
        "ROC-AUC": roc_auc_score(y_test, probability),
        "Average precision": average_precision_score(y_test, probability),
        "Sensitivity": recall_score(y_test, prediction, zero_division=0),
        "Specificity": tn / (tn + fp),
        "Precision": precision_score(y_test, prediction, zero_division=0),
        "F1": f1_score(y_test, prediction, zero_division=0),
        "Brier score": brier_score_loss(y_test, probability),
    })
    xgb_fit_rows.append({
        "Fold": fold, "Train nhãn 0": n_negative, "Train nhãn 1": n_positive,
        "scale_pos_weight": scale_pos_weight, "Fit pipeline (giây)": fit_seconds,
    })
    xgb_baseline_models[fold] = model
xgb_fold_metrics = pd.DataFrame(xgb_fold_rows).set_index("Fold")
xgb_fold_metrics.round(4)

,ROC-AUC,Average precision,Sensitivity,Specificity,Precision,F1,Brier score
Fold,,,,,,,
1,0.7833,0.4594,0.7758,0.6449,0.3673,0.4986,0.1940
2,0.7969,0.4894,0.7802,0.6735,0.3884,0.5186,0.1870
3,0.8045,0.4882,0.8194,0.6466,0.3808,0.5199,0.1881
4,0.8059,0.4759,0.7890,0.6873,0.4016,0.5322,0.1815
5,0.8097,0.4898,0.8198,0.6680,0.3964,0.5344,0.1821


In [17]:
# Thời gian gồm cả fit tiền xử lý và classifier, không gồm predict.
xgb_fit_times = pd.DataFrame(xgb_fit_rows).set_index("Fold")
xgb_fit_times.round({"scale_pos_weight": 6, "Fit pipeline (giây)": 3})

,Train nhãn 0,Train nhãn 1,scale_pos_weight,Fit pipeline (giây)
Fold,,,,
1,6846,1819,3.763606,0.153
2,6846,1819,3.763606,0.076
3,6846,1820,3.761538,0.078
4,6847,1819,3.764156,0.078
5,6847,1819,3.764156,0.084


In [18]:
xgb_metric_summary = xgb_fold_metrics.agg(["mean", "std"]).rename(
    index={"mean": "Trung bình", "std": "Độ lệch chuẩn"}
)
xgb_metric_summary.round(4)

,ROC-AUC,Average precision,Sensitivity,Specificity,Precision,F1,Brier score
Trung bình,0.8000,0.4805,0.7968,0.6641,0.3869,0.5208,0.1866
Độ lệch chuẩn,0.0105,0.0131,0.0213,0.0182,0.0135,0.0143,0.0051


In [19]:
baseline_comparison = pd.DataFrame({
    name: metrics.agg(["mean", "std"]).T.apply(
        lambda row: f"{row['mean']:.4f} ± {row['std']:.4f}", axis=1
    )
    for name, metrics in [("LR", lr_fold_metrics), ("XGBoost", xgb_fold_metrics)]
})
baseline_comparison.index.name = "Metric"
baseline_comparison

,LR,XGBoost
Metric,,
ROC-AUC,0.7953 ± 0.0121,0.8000 ± 0.0105
Average precision,0.4642 ± 0.0208,0.4805 ± 0.0131
Sensitivity,0.7696 ± 0.0222,0.7968 ± 0.0213
Specificity,0.6845 ± 0.0112,0.6641 ± 0.0182
Precision,0.3934 ± 0.0146,0.3869 ± 0.0135
F1,0.5206 ± 0.0177,0.5208 ± 0.0143
Brier score,0.1891 ± 0.0060,0.1866 ± 0.0051


In [20]:
assert xgb_oof_probability.index.equals(X.index) and xgb_oof_probability.index.equals(y.index)
assert xgb_oof_probability.notna().all() and xgb_oof_fold.notna().all()
assert np.isfinite(xgb_oof_probability).all()
assert xgb_oof_probability.between(0, 1).all()
assert np.all(xgb_oof_coverage == 1)
xgb_oof_prediction = xgb_oof_probability.ge(0.5).astype("Int64").rename("prediction")
xgb_oof = pd.DataFrame({
    "SEQN": support["SEQN"], "cycle": support["cycle"],
    "y_true": y, "probability_1": xgb_oof_probability,
    "prediction": xgb_oof_prediction, "outer_fold": xgb_oof_fold,
})
assert xgb_oof.index.equals(X.index) and xgb_oof["SEQN"].is_unique
assert len(xgb_oof) == 10832
pd.testing.assert_series_equal(xgb_oof["outer_fold"], lr_oof["outer_fold"])
pd.testing.assert_frame_equal(xgb_oof[["SEQN", "cycle", "y_true"]], lr_oof[["SEQN", "cycle", "y_true"]])
for fold, (_, test_idx) in enumerate(outer_splits, start=1):
    assert xgb_oof_fold.iloc[test_idx].eq(fold).all()
    assert set(xgb_oof.loc[xgb_oof["outer_fold"].eq(fold), "SEQN"]) == set(support.iloc[test_idx]["SEQN"])
xgb_confusion = pd.DataFrame(
    confusion_matrix(y, xgb_oof_prediction, labels=[0, 1]),
    index=pd.Index(["0", "1"], name="Nhãn thật"),
    columns=pd.Index(["0", "1"], name="Dự đoán (ngưỡng 0,5)"),
)
xgb_confusion

"Dự đoán (ngưỡng 0,5)",0,1
Nhãn thật,,
0,5683,2875
1,462,1812


In [21]:
pd.testing.assert_frame_equal(X, X_before_xgb_baseline)
pd.testing.assert_series_equal(y, y_before_xgb_baseline)
assert len({id(m.named_steps["preprocessor"]) for m in xgb_baseline_models.values()}) == 5
assert len({id(m.named_steps["classifier"]) for m in xgb_baseline_models.values()}) == 5
xgb_baseline_warnings = pd.DataFrame(xgb_warning_rows, columns=["Fold", "Loại", "Thông báo"])
pd.Series({
    "Số dự đoán ngoài fold": len(xgb_oof_probability),
    "Dự đoán thiếu": int(xgb_oof_probability.isna().sum()),
    "Mỗi người được dự đoán đúng một lần": bool(np.all(xgb_oof_coverage == 1)),
    "Cùng người và fold với LR": True,
    "X/y giữ nguyên": True,
    "Số cảnh báo XGBoost": len(xgb_baseline_warnings),
}, name="Kiểm tra").to_frame()

,Kiểm tra
Số dự đoán ngoài fold,10832
Dự đoán thiếu,0
Mỗi người được dự đoán đúng một lần,True
Cùng người và fold với LR,True
X/y giữ nguyên,True
Số cảnh báo XGBoost,0


In [22]:
xgb_baseline_warnings

,Fold,Loại,Thông báo


Thời gian fit tính cả tiền xử lý và phụ thuộc máy chạy. Bảng LR/XGBoost dùng trung bình ± SD giữa 5 fold.

## RF và SVM baseline

Dùng lại X, y, `outer_splits` và kết quả LR/XGBoost đã có. RF dùng 300 cây, `max_depth=None`, `min_samples_leaf=5`, `max_features="sqrt"`, `n_jobs=4`; SVM dùng RBF với `C=1`, `gamma="scale"`, `probability=True`. Cả hai dùng `class_weight="balanced"` từ y_train, `random_state=42`; mỗi fold tạo pipeline mới, RF không scaling và SVM có scaling.

RF dùng xác suất cho ROC-AUC/AP, Brier và ngưỡng phân loại 0,5. SVM dùng `decision_function` cho ROC-AUC/AP và phân loại nhãn 1 khi score ≥0; `predict_proba` dùng cho Brier. `probability=True` có bước ước lượng xác suất nội bộ, nên score = 0 không đồng nghĩa xác suất = 0,5.

In [23]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

required_baseline_results = ["lr_fold_metrics", "xgb_fold_metrics", "lr_oof", "xgb_oof"]
missing_baseline_results = [name for name in required_baseline_results if name not in globals()]
if missing_baseline_results:
    raise RuntimeError(f"Cần chạy các ô LR/XGBoost trước: {missing_baseline_results}")
for old_oof in [lr_oof, xgb_oof]:
    assert old_oof.index.equals(X.index) and old_oof.index.equals(y.index)
    pd.testing.assert_series_equal(old_oof["SEQN"], support["SEQN"])
    pd.testing.assert_series_equal(old_oof["y_true"], y.rename("y_true"))
    for fold, (_, test_idx) in enumerate(outer_splits, start=1):
        assert old_oof["outer_fold"].iloc[test_idx].eq(fold).all()
assert lr_fold_metrics.index.tolist() == xgb_fold_metrics.index.tolist() == [1, 2, 3, 4, 5]
assert np.isfinite(lr_fold_metrics.to_numpy()).all() and np.isfinite(xgb_fold_metrics.to_numpy()).all()

X_before_rf_svm = X.copy(deep=True)
y_before_rf_svm = y.copy(deep=True)
rf_svm_configs = {
    "RF": (False, RandomForestClassifier, {
        "n_estimators": 300, "max_depth": None, "min_samples_leaf": 5,
        "max_features": "sqrt", "class_weight": "balanced",
        "random_state": 42, "n_jobs": 4,
    }),
    "SVM RBF": (True, SVC, {
        "C": 1.0, "kernel": "rbf", "gamma": "scale",
        "class_weight": "balanced", "probability": True, "random_state": 42,
    }),
}

In [24]:
rf_svm_models = {}
rf_svm_oof = {}
rf_svm_fold_metrics = {}
rf_svm_fit_rows = []
rf_svm_warning_rows = []
rf_svm_check_rows = []
for model_name, (scale_numeric, classifier_type, params) in rf_svm_configs.items():
    oof_probability = pd.Series(np.nan, index=X.index, name="probability_1")
    oof_score = pd.Series(np.nan, index=X.index, name="score")
    oof_prediction = pd.Series(pd.NA, index=X.index, dtype="Int64", name="prediction")
    oof_fold = pd.Series(pd.NA, index=X.index, dtype="Int64", name="outer_fold")
    coverage = np.zeros(len(X), dtype=int)
    fold_rows = []
    rf_svm_models[model_name] = {}
    for fold, (train_idx, test_idx) in enumerate(outer_splits, start=1):
        X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
        X_test, y_test = X.iloc[test_idx], y.iloc[test_idx]
        model = Pipeline([
            ("preprocessor", build_preprocessor(scale_numeric=scale_numeric)),
            ("classifier", classifier_type(**params)),
        ])
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            fit_start = perf_counter()
            model.fit(X_train, y_train)
            fit_seconds = perf_counter() - fit_start
        for warning in caught:
            rf_svm_warning_rows.append({
                "Mô hình": model_name, "Fold": fold, "Loại": warning.category.__name__,
                "Thông báo": str(warning.message),
                "Không hội tụ": issubclass(warning.category, ConvergenceWarning),
            })
        classifier = model.named_steps["classifier"]
        assert classifier.class_weight == "balanced"
        np.testing.assert_array_equal(model.classes_, [0, 1])
        probability = model.predict_proba(X_test)[:, 1]
        if model_name == "SVM RBF":
            score = model.decision_function(X_test)
            prediction = (score >= 0).astype(int)
            expected_weights = compute_class_weight("balanced", classes=np.array([0, 1]), y=y_train.to_numpy(dtype=int))
            np.testing.assert_allclose(classifier.class_weight_, expected_weights)
            assert classifier.fit_status_ == 0
        else:
            score = probability
            prediction = (probability >= 0.5).astype(int)
        assert np.isfinite(score).all() and np.isfinite(probability).all()
        assert ((probability >= 0) & (probability <= 1)).all()
        oof_probability.iloc[test_idx] = probability
        oof_score.iloc[test_idx] = score
        oof_prediction.iloc[test_idx] = prediction
        oof_fold.iloc[test_idx] = fold
        coverage[test_idx] += 1
        tn, fp, fn, tp = confusion_matrix(y_test, prediction, labels=[0, 1]).ravel()
        fold_rows.append({
            "Fold": fold,
            "ROC-AUC": roc_auc_score(y_test, score),
            "Average precision": average_precision_score(y_test, score),
            "Sensitivity": recall_score(y_test, prediction, zero_division=0),
            "Specificity": tn / (tn + fp),
            "Precision": precision_score(y_test, prediction, zero_division=0),
            "F1": f1_score(y_test, prediction, zero_division=0),
            "Brier score": brier_score_loss(y_test, probability),
        })
        rf_svm_fit_rows.append({"Mô hình": model_name, "Fold": fold, "Fit pipeline (giây)": fit_seconds})
        rf_svm_models[model_name][fold] = model
    assert np.all(coverage == 1)
    assert oof_probability.notna().all() and oof_score.notna().all() and oof_prediction.notna().all()
    result = pd.DataFrame({
        "SEQN": support["SEQN"], "cycle": support["cycle"], "y_true": y,
        "score": oof_score, "probability_1": oof_probability,
        "prediction": oof_prediction, "outer_fold": oof_fold,
    })
    assert result.index.equals(X.index) and result.index.equals(y.index)
    assert result["SEQN"].is_unique and len(result) == 10832
    pd.testing.assert_frame_equal(result[["SEQN", "cycle", "y_true", "outer_fold"]], lr_oof[["SEQN", "cycle", "y_true", "outer_fold"]])
    rf_svm_oof[model_name] = result
    rf_svm_fold_metrics[model_name] = pd.DataFrame(fold_rows).set_index("Fold")
    rf_svm_check_rows.append({
        "Mô hình": model_name, "Số dự đoán": len(result),
        "Dự đoán thiếu": int(result["probability_1"].isna().sum()),
        "Đúng một lần/người": bool(np.all(coverage == 1)),
        "Cùng người/fold với LR": True,
    })
rf_fold_metrics = rf_svm_fold_metrics["RF"]
svm_fold_metrics = rf_svm_fold_metrics["SVM RBF"]
rf_oof = rf_svm_oof["RF"]
svm_oof = rf_svm_oof["SVM RBF"]
pd.concat(rf_svm_fold_metrics, names=["Mô hình"]).round(4)

ROC-AUC  Average precision  Sensitivity  Specificity  Precision  \
Mô hình Fold                                                                    
RF      1      0.7683             0.4249       0.7121       0.6805     0.3720   
        2      0.7794             0.4448       0.6989       0.7120     0.3921   
        3      0.7929             0.4588       0.7511       0.6904     0.3915   
        4      0.7835             0.4365       0.6571       0.7218     0.3858   
        5      0.7932             0.4606       0.7253       0.7171     0.4054   
SVM RBF 1      0.7773             0.4264       0.7912       0.6367     0.3666   
        2      0.7822             0.4354       0.7890       0.6478     0.3732   
        3      0.7922             0.4491       0.8392       0.6373     0.3802   
        4      0.7879             0.4329       0.7978       0.6581     0.3829   
        5      0.8054             0.4757       0.8330       0.6458     0.3848   

                  F1  Brier score  
Mô hình Fold                       
RF      1     0.4887       0.1872  
        2     0.5024       0.1805  
        3     0.5147       0.1800  
        4     0.4862       0.1764  
        5     0.5201       0.1755  
SVM RBF 1     0.5010       0.1402  
        2     0.5067       0.1387  
        3     0.5234       0.1361  
        4     0.5175       0.1379  
        5     0.5264       0.1329

In [25]:
rf_metric_summary = rf_fold_metrics.agg(["mean", "std"])
svm_metric_summary = svm_fold_metrics.agg(["mean", "std"])
pd.concat({
    "RF": rf_metric_summary, "SVM RBF": svm_metric_summary,
}, names=["Mô hình"]).rename(index={"mean": "Trung bình", "std": "Độ lệch chuẩn"}).round(4)

ROC-AUC  Average precision  Sensitivity  Specificity  \
Mô hình                                                                       
RF      Trung bình      0.7834             0.4451       0.7089       0.7044   
        Độ lệch chuẩn   0.0104             0.0151       0.0348       0.0180   
SVM RBF Trung bình      0.7890             0.4439       0.8100       0.6451   
        Độ lệch chuẩn   0.0108             0.0196       0.0241       0.0088   

                       Precision      F1  Brier score  
Mô hình                                                
RF      Trung bình        0.3894  0.5024       0.1799  
        Độ lệch chuẩn     0.0121  0.0151       0.0046  
SVM RBF Trung bình        0.3775  0.5150       0.1372  
        Độ lệch chuẩn     0.0075  0.0108       0.0028

In [26]:
rf_svm_fit_times = pd.DataFrame(rf_svm_fit_rows).pivot(
    index="Fold", columns="Mô hình", values="Fit pipeline (giây)"
)
rf_svm_fit_times.round(3)

Mô hình,RF,SVM RBF
Fold,,
1,0.491,7.499
2,0.480,7.343
3,0.480,7.386
4,0.470,7.407
5,0.490,7.520


In [27]:
rf_svm_confusion = pd.concat({
    name: pd.DataFrame(
        confusion_matrix(y, result["prediction"], labels=[0, 1]),
        index=pd.Index(["0", "1"], name="Nhãn thật"),
        columns=pd.Index(["0", "1"], name="Dự đoán"),
    )
    for name, result in rf_svm_oof.items()
}, names=["Mô hình"])
rf_svm_confusion

Dự đoán               0     1
Mô hình Nhãn thật            
RF      0          6028  2530
        1           662  1612
SVM RBF 0          5521  3037
        1           432  1842

In [28]:
all_baseline_metrics = {
    "LR": lr_fold_metrics, "XGBoost": xgb_fold_metrics,
    "RF": rf_fold_metrics, "SVM RBF": svm_fold_metrics,
}
baseline_comparison = pd.DataFrame({
    name: metrics.agg(["mean", "std"]).T.apply(
        lambda row: f"{row['mean']:.4f} ± {row['std']:.4f}", axis=1
    )
    for name, metrics in all_baseline_metrics.items()
})
baseline_comparison.index.name = "Metric"
baseline_comparison

,LR,XGBoost,RF,SVM RBF
Metric,,,,
ROC-AUC,0.7953 ± 0.0121,0.8000 ± 0.0105,0.7834 ± 0.0104,0.7890 ± 0.0108
Average precision,0.4642 ± 0.0208,0.4805 ± 0.0131,0.4451 ± 0.0151,0.4439 ± 0.0196
Sensitivity,0.7696 ± 0.0222,0.7968 ± 0.0213,0.7089 ± 0.0348,0.8100 ± 0.0241
Specificity,0.6845 ± 0.0112,0.6641 ± 0.0182,0.7044 ± 0.0180,0.6451 ± 0.0088
Precision,0.3934 ± 0.0146,0.3869 ± 0.0135,0.3894 ± 0.0121,0.3775 ± 0.0075
F1,0.5206 ± 0.0177,0.5208 ± 0.0143,0.5024 ± 0.0151,0.5150 ± 0.0108
Brier score,0.1891 ± 0.0060,0.1866 ± 0.0051,0.1799 ± 0.0046,0.1372 ± 0.0028


In [29]:
pd.testing.assert_frame_equal(X, X_before_rf_svm)
pd.testing.assert_series_equal(y, y_before_rf_svm)
all_new_models = [model for models in rf_svm_models.values() for model in models.values()]
assert len({id(m.named_steps["preprocessor"]) for m in all_new_models}) == 10
assert len({id(m.named_steps["classifier"]) for m in all_new_models}) == 10
rf_svm_warnings = pd.DataFrame(rf_svm_warning_rows, columns=[
    "Mô hình", "Fold", "Loại", "Thông báo", "Không hội tụ",
])
rf_svm_checks = pd.DataFrame(rf_svm_check_rows).set_index("Mô hình")
rf_svm_checks["X/y không đổi"] = True
rf_svm_checks["Cảnh báo không hội tụ"] = [
    sum(row["Không hội tụ"] for row in rf_svm_warning_rows if row["Mô hình"] == name)
    for name in rf_svm_checks.index
]
rf_svm_checks

,Số dự đoán,Dự đoán thiếu,Đúng một lần/người,Cùng người/fold với LR,X/y không đổi,Cảnh báo không hội tụ
Mô hình,,,,,,
RF,10832,0,True,True,True,0
SVM RBF,10832,0,True,True,True,0


In [30]:
rf_svm_warnings

,Mô hình,Fold,Loại,Thông báo,Không hội tụ
0,SVM RBF,1,FutureWarning,The `probability` parameter was deprecated in ...,False
1,SVM RBF,2,FutureWarning,The `probability` parameter was deprecated in ...,False
2,SVM RBF,3,FutureWarning,The `probability` parameter was deprecated in ...,False
3,SVM RBF,4,FutureWarning,The `probability` parameter was deprecated in ...,False
4,SVM RBF,5,FutureWarning,The `probability` parameter was deprecated in ...,False


Bảng so sánh gồm bốn baseline cố định, chưa dùng để chốt mô hình tốt nhất. Thời gian fit SVM tính cả bước ước lượng xác suất nội bộ; không có bước calibration bổ sung hay đánh giá BRFSS trong notebook này.